# Preprocesamiento de Datos a Formato YOLO

El objetivo de este notebook es unificar y preparar los datos crudos originales (imagenes en formato `.jpg` y anotaciones en formato `XML` tipo Pascal VOC) para el entrenamiento de nuestro modelo en arquitecturas **YOLO contemporaneas**.

## Contexto y justificativo
Las arquitecturas YOLO no ingieren archivos XML originarios de PASCAL de forma nativa. Para que la generacion de datasets sea compatible, YOLO exige que cada imagen venga acompanada de un **archivo de texto plano (`.txt`)** de identico nombre. Adicional a este texto, el pipeline de YOLO precisa que nosotros structuremos internamente las carpetas dividiendolas por los subgrupos a someter al entrenamiento, por ejemplo `train/`, `val/`, `test/`.

## El formato YOLO
Si una unica imagen cuenta con uno o mas cromosomas (objetos), cada linea del archivo `.txt` va a representar a un objeto delimitado individual, respetando estricta y rigurosamente este orden espacial, separado por espacios:

`<class_id> <x_center> <y_center> <width> <height>`

## ¿Por que se requiere normalizar las coordenadas?
El algoritmo YOLO postula que todo pixel posicional (`x_center`, `y_center`) y toda proporcion geometrica envolvente del *bounding box* (`width`, `height`) este matematizada en **intervalos estrictamente normalizados del `0.0` al `1.0`.** 

- **¿Como logramos esto?** Dividimos la coordenada en el eje X de tu limite por la anchura final de la imagen (`width`), y hacemos lo mismo para el eje Y respecto a su altura (`height`).
- **¿Por que asi?** Por que los valores flotantes relativos vuelven al modelo cien veces mas rapido para computar back-propagation por invariancia matemantica de resoluciones: Las imagenes se manipulan y pueden alterarse durante el *Data Augmentation* en resoluciones variables al entrenamiento, y las etiquetas con proporciones relativas van dictando el limite de la caja envolvente siempre preciso de forma universal sin alterar recalculos. Ademas provee al pre-procesador tensor una barrera para no sufrir colapsos por re-escalados inorportunos.

---
## 1. Importacion de Librerias y Definicion de la Estructura de Clases
Empezamos consolidando rutas universales hasta nuestra data pre-versionada y construyendo el mapeo exhaustivo de los tags para las 24 clases cromosomicas objetivo.

In [1]:
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split

# Rutas principales
BASE_DIR = Path('../../')
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
IMAGES_DIR = RAW_DATA_DIR / 'JEPG'
ANNOTATIONS_DIR = RAW_DATA_DIR / 'annotations'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Clases especificadas
CLASSES = [
    'A1', 'A2', 'A3', 'B4', 'B5', 'C10', 'C11', 'C12', 'C6', 'C7', 'C8', 'C9',
    'D13', 'D14', 'D15', 'E16', 'E17', 'E18', 'F19', 'F20', 'G21', 'G22', 'X', 'Y'
]
CLASS_2_ID = {cls: i for i, cls in enumerate(CLASSES)}

## 2. Recopilacion General y Particion de Splits de Data

Una vez escaneamos y recopilamos validamente los listados cruzados de imagen y xml, haremos un particionamiento de nuestro volumen en **tres fracciones obligatorias**:

1. **Train (Entrenamiento) 70%**.
2. **Val (Validacion) 15%**.
3. **Test (Prueba) 15%**.


In [2]:
# Recopilar pares validos de IMAGEN -> XML
xml_files = list(ANNOTATIONS_DIR.glob('*.xml'))
valid_data = []

for xml_path in xml_files:
    img_path = IMAGES_DIR / f"{xml_path.stem}.jpg"
    if img_path.exists():
        valid_data.append((xml_path, img_path))

print(f"Total de pares (imagen, xml) validos: {len(valid_data)}")

# Realizar particion 70/15/15 con sklearn
# 1. Separamos el 70% para train y el 30% restante para (val + test)
train_data, temp_data = train_test_split(valid_data, test_size=0.30, random_state=42)

# 2. Separamos el 30% restante por la mitad, obteniendo 15% val y 15% test
val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42)

splits = {
    'train': train_data,
    'val': val_data,
    'test': test_data
}

print("Distribucion de datos:")
print(f" - Train (70%): {len(splits['train'])} imagenes")
print(f" - Val   (15%): {len(splits['val'])} imagenes")
print(f" - Test  (15%): {len(splits['test'])} imagenes")

Total de pares (imagen, xml) validos: 5000
Distribucion de datos:
 - Train (70%): 3500 imagenes
 - Val   (15%): 750 imagenes
 - Test  (15%): 750 imagenes


## 3. Construccion de Bounding Boxes hacia Estructuras YOLO

Este es el corazon real del notebook en donde, con todo fraccionado, construimos la carpeta `data/processed`:

1. **Parses de geometria**: Vamos iterando un documento `XML` hasta extraer tuplas envolventes minimas (`xmin, ymin, xmax, ymax`).
2. **Refinado**: Reestrucuramos hacia variables YOLO: calculo del punto central (`x_center, y_center`), y extraccion de las distancias horizontales-verticales (`box_w, box_h`), luego divididos por la resolucion normal originaria (`image width / height`).
3. **Clipping Protector / Control de excepciones**: Ocasionalmente y por manipuladores humanos, ciertas cajas sobrepasan el dimensionamiento final del eje delimitador logico (e.g. `Width max`), dejando ratios `1.012`. Esto interrumpe el *dataset loader* de ultralytics. Realizamos una contension rigurosa por minimos locales `min(1.0, value)` blindando su pureza.
4. **Almacenamiento fisico**: Transferimos en texto las clases hacia  `{split}/labels/*.txt`  alojando fisicamente, como pares, la fotografa inicial a `{split}/images/*.jpg`.

In [ ]:
# Diccionario para rastrear clases desconocidas
unknown_classes = {}

# Lista para rastrear valores normalizados recortados (fuera de bordes)
clipped_boxes = []

# Crear directorio de procesamiento, convertir anotaciones a YOLO y copiar imagenes
for split_name, data_pairs in splits.items():
    images_out_dir = PROCESSED_DIR / split_name / 'images'
    labels_out_dir = PROCESSED_DIR / split_name / 'labels'
    
    images_out_dir.mkdir(parents=True, exist_ok=True)
    labels_out_dir.mkdir(parents=True, exist_ok=True)
    
    for xml_path, img_path in data_pairs:
        # Abrir archivo XML
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        size_elem = root.find("size")
        if size_elem is None: continue
        width = int(size_elem.findtext("width", "0"))
        height = int(size_elem.findtext("height", "0"))
        if width == 0 or height == 0: continue
        
        yolo_lines = []
        for obj in root.findall("object"):
            cls_name = obj.findtext("name")
            if cls_name not in CLASS_2_ID:
                unknown_classes[cls_name] = unknown_classes.get(cls_name, 0) + 1
                continue
            
            cls_id = CLASS_2_ID[cls_name]
            bndbox = obj.find("bndbox")
            xmin = float(bndbox.findtext("xmin", "0"))
            ymin = float(bndbox.findtext("ymin", "0"))
            xmax = float(bndbox.findtext("xmax", "0"))
            ymax = float(bndbox.findtext("ymax", "0"))
            
            x_center = ((xmin + xmax) / 2.0) / width
            y_center = ((ymin + ymax) / 2.0) / height
            box_w = (xmax - xmin) / width
            box_h = (ymax - ymin) / height
            
            if x_center < 0.0 or x_center > 1.0 or y_center < 0.0 or y_center > 1.0 or box_w < 0.0 or box_w > 1.0 or box_h < 0.0 or box_h > 1.0:
                clipped_boxes.append((xml_path.name, cls_name, x_center, y_center, box_w, box_h))
            
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            box_w = max(0.0, min(1.0, box_w))
            box_h = max(0.0, min(1.0, box_h))
            
            yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
            
        # Escribir archivo txt de anotaciones
        txt_out_path = labels_out_dir / f"{xml_path.stem}.txt"
        with open(txt_out_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(yolo_lines))
            
        img_out_path = images_out_dir / img_path.name
        if not img_out_path.exists():
            shutil.copy(img_path, img_out_path)

print("Transformacion XML -> YOLO y organizacion del conjunto finalizados con exito!")

# Reporte de validacion de clases
if unknown_classes:
    print("\n¡ Se encontraron las siguientes clases fuera de las originales:")
    for cls, count in unknown_classes.items():
        print(f" - Clase '{cls}': encontrada {count} veces (ignorada).")
else:
    print("\nTodas las clases evaluadas pertenecen de forma estricta a la lista enviada original.")

# Reporte de coordenadas fuera de limites (recortadas)
if clipped_boxes:
    print("\n¡Se encontraron y recortaron coordenadas que excedian los limites de la imagen [0.0 - 1.0]:")
    for entry in clipped_boxes[:10]:  # Mostrar maximo 10 ejemplos
        print(f" - Archivo: {entry[0]} | Clase: {entry[1]} | (xc, yc, w, h) antes del recorte: ({entry[2]:.4f}, {entry[3]:.4f}, {entry[4]:.4f}, {entry[5]:.4f})")
    if len(clipped_boxes) > 10:
        print(f"   ... y {len(clipped_boxes) - 10} casos mas.")
else:
    print("\nTodas las coordenadas normalizadas encajaron perfectamentente dentro de los limites de la imagen.")

Transformacion XML -> YOLO y organizacion del conjunto finalizados con exito!

Todas las clases evaluadas pertenecen de forma estricta a la lista enviada original.

Todas las coordenadas normalizadas encajaron perfectamentente dentro de los limites de la imagen.


## 4. Archivo Manifiesto Base (data.yml)

Un algoritmo YOLO no intenta recorrer subcarpetas intuitivamente sin guias y desconoce tu arreglo original formativo; YOLO confia por default en buscar de inmediato un fichero configurativo o manifiesto estatico denominado historicamente **data manifest / yaml**.

Este bloque fabrica programaticamente tu nucleo de arranque especificandole a la red neuronal tres caracteristicas de oro:
- **`path`, `train`, `val`, `test`**: Directorios desde los raices donde arranca tu inferencia de red y con respecto a las colecciones directas recien alojadas, consumiendo la franja `images/` de cada seccion.
- **`nc` (Number of Classes)**: La cantidad formal de etiquetas que debe inicializarse para las capas finales del modelo de red, que sera de `24`.
- **`names`**: Arreglo exacto con el diccionario universal ordenado indexadamente a las variables textuales para clasificacion humana cromosomica.

In [4]:
# Generar archivo data.yml exigido en raiz para entrenar con YOLO
yaml_content = f"""path: ../data/processed  # Ruta desde donde sea invocado comunmente tu notebook/train
train: train/images
val: val/images
test: test/images

nc: {len(CLASSES)}
names:
"""

for c in CLASSES:
    yaml_content += f"  - {c}\n"

yaml_file = PROCESSED_DIR / 'data.yml'
with open(yaml_file, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"Archivo de configuracion creado exitosamente en {yaml_file}\n")
print(yaml_content)

Archivo de configuracion creado exitosamente en ..\..\data\processed\data.yml

path: ../data/processed  # Ruta desde donde sea invocado comunmente tu notebook/train
train: train/images
val: val/images
test: test/images

nc: 24
names:
  - A1
  - A2
  - A3
  - B4
  - B5
  - C10
  - C11
  - C12
  - C6
  - C7
  - C8
  - C9
  - D13
  - D14
  - D15
  - E16
  - E17
  - E18
  - F19
  - F20
  - G21
  - G22
  - X
  - Y

